# Entraîner « ok nao » (openWakeWord, voix françaises)

À ouvrir dans Google Colab avec un GPU (*Exécution > Modifier le type d'exécution > GPU T4*).
Compter 1 à 2 h : téléchargements (~20 Go), voix de synthèse, augmentation, entraînement.

Résultat : `ok_nao.onnx`, à copier sur le drone dans
`robotics/drones/static/onboard/models/` (voir `models/README.md`).

## 1. Code et dépendances

In [ ]:
!git clone -q https://github.com/Monolithoid/ia-agent.git
!git clone -q https://github.com/dscripka/openWakeWord.git
!pip install -q --no-deps -e ./openWakeWord
!pip install -q onnxruntime onnx onnxscript scipy scikit-learn tqdm requests pyyaml \
    mutagen torchinfo torchmetrics speechbrain==0.5.14 audiomentations torch-audiomentations \
    acoustics pronouncing "datasets<3" edge-tts piper-tts
!apt-get -qq install -y ffmpeg > /dev/null
# train.py importe generate_samples (générateur anglais) : module vide à la place
!mkdir -p stub && echo "def generate_samples(*a, **k): raise SystemExit('clips : generate_clips_fr.py')" > stub/generate_samples.py
!cp ia-agent/robotics/drones/static/onboard/models/ok_nao.yml .

## 2. Voix de synthèse françaises

edge-tts (voix fr-FR, fr-CA, fr-BE, fr-CH) et Piper (dont `fr_FR-mls-medium`, 125 locuteurs).
Négatifs : mots proches (« ok », « nao », « cacao », « ok non »…).

In [ ]:
!python ia-agent/robotics/drones/static/onboard/models/generate_clips_fr.py \
    --out sortie/ok_nao --n-train 3000 --n-test 300

## 3. Réverbération, bruits de fond, négatifs précalculés

In [ ]:
import os, numpy as np, scipy.io.wavfile, datasets
from tqdm import tqdm

# Réponses impulsionnelles de pièces (MIT)
os.makedirs("mit_rirs", exist_ok=True)
for i, row in enumerate(tqdm(datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses",
                                                    split="train", streaming=True))):
    scipy.io.wavfile.write(f"mit_rirs/{i}.wav", 16000,
                           (row["audio"]["array"] * 32767).astype(np.int16))

In [ ]:
# Bruits de fond : un morceau d'AudioSet et ~1 h de musique (FMA), en 16 kHz
!mkdir -p audioset audioset_16k fma
!wget -q -O audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/bal_train09.tar
!tar -xf audioset/bal_train09.tar -C audioset
import glob
for f in tqdm(glob.glob("audioset/**/*.flac", recursive=True)):
    out = "audioset_16k/" + os.path.basename(f).replace(".flac", ".wav")
    os.system(f'ffmpeg -loglevel error -y -i "{f}" -ac 1 -ar 16000 "{out}"')

fma = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma = iter(fma.cast_column("audio", datasets.Audio(sampling_rate=16000)))
for i in tqdm(range(120)):           # 120 extraits de 30 s
    row = next(fma)
    scipy.io.wavfile.write(f"fma/{i}.wav", 16000, (row["audio"]["array"] * 32767).astype(np.int16))

In [ ]:
# Négatifs précalculés (~2000 h) et jeu de validation des faux déclenchements (~11 h)
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

## 4. Augmentation et entraînement

Pas de `--generate_clips` : les clips de l'étape 2 sont déjà dans `sortie/ok_nao`.
La conversion finale en `.tflite` peut échouer (TensorFlow absent) : seul `ok_nao.onnx` sert au drone.

In [ ]:
!python -m openwakeword.train --training_config ok_nao.yml --augment_clips
!python -m openwakeword.train --training_config ok_nao.yml --train_model || true
!ls -la sortie/ok_nao.onnx

## 5. Essai rapide puis téléchargement

In [ ]:
from openwakeword.model import Model
import openwakeword.utils, glob
openwakeword.utils.download_models(["ok_nao"])          # modèles communs (mel, embedding)
m = Model(wakeword_models=["sortie/ok_nao.onnx"], inference_framework="onnx")
for kind in ("positive_test", "negative_test"):
    clips = glob.glob(f"sortie/ok_nao/{kind}/*.wav")[:200]
    hits = sum(max(p["ok_nao"] for p in m.predict_clip(c)) >= 0.5 for c in clips)
    print(f"{kind} : {hits}/{len(clips)} au-dessus de 0,5")

In [ ]:
from google.colab import files
files.download("sortie/ok_nao.onnx")